In [ ]:
"""
ProtVec-based ML Classification Pipeline (Enhanced with Multi-Model Comparison)
For: merged_output.xlsx (training) + 38-experimental-validation.xlsx (external validation)
Output: C:\Temp\database\fingerprint
Models: LogisticRegression, RandomForest, SVM, LightGBM, XGBoost
Features: Multi-model comparison, ROC curves, Confusion matrices, Model overview
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc, 
    roc_auc_score, accuracy_score, precision_recall_curve, f1_score,
    precision_score, recall_score, balanced_accuracy_score, matthews_corrcoef
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

# Try to import LightGBM and XGBoost (optional)
try:
    import lightgbm as lgb
    HAS_LGBM = True
except:
    HAS_LGBM = False
    print("[WARNING] LightGBM not installed, skipping LightGBM model")

try:
    import xgboost as xgb
    HAS_XGB = True
except:
    HAS_XGB = False
    print("[WARNING] XGBoost not installed, skipping XGBoost model")

# ============================================================================
# ======================== ROC VISUALIZATION CLASS ===========================
# ============================================================================

class ROCVisualization:
    """Visualize ROC curves for multi-class classification."""
    
    def __init__(self, colors=None):
        self.colors = colors or sns.color_palette("husl", 10)
    
    def plot_macro_roc_curve(self, y_true, y_pred_proba, class_names, 
                            title, output_path, dpi=300):
        """Plot macro-averaged ROC curve for multi-class classification."""
        
        n_classes = len(class_names)
        y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))
        
        fpr_list = []
        tpr_list = []
        auc_per_class = {}
        
        for i in range(n_classes):
            if y_true_bin[:, i].sum() == 0:
                print(f"    ⚠️  No positive samples for class {class_names[i]}")
                continue
            
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
            roc_auc = auc(fpr, tpr)
            auc_per_class[class_names[i]] = roc_auc
            fpr_list.append(fpr)
            tpr_list.append(tpr)
        
        mean_fpr = np.linspace(0, 1, 100)
        tpr_interp_list = []
        
        for fpr, tpr in zip(fpr_list, tpr_list):
            tpr_interp = np.interp(mean_fpr, fpr, tpr)
            tpr_interp_list.append(tpr_interp)
        
        mean_tpr = np.mean(tpr_interp_list, axis=0)
        mean_tpr[0], mean_tpr[-1] = 0, 1
        macro_auc = auc(mean_fpr, mean_tpr)
        
        fig, ax = plt.subplots(figsize=(10, 8), dpi=dpi)
        
        ax.plot(mean_fpr, mean_tpr, 
                label=f'Macro-avg ROC (AUC = {macro_auc:.3f})', 
                lw=3.5, color='darkblue')
        ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
        
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('False Positive Rate', fontsize=13, fontweight='bold')
        ax.set_ylabel('True Positive Rate', fontsize=13, fontweight='bold')
        ax.set_title(title, fontsize=15, fontweight='bold')
        ax.legend(loc='lower right', fontsize=12)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=dpi, bbox_inches='tight')
        plt.close()
        
        roc_csv_filename = output_path.stem.replace('roc_', 'roc_data_') + '.csv'
        roc_csv_path = output_path.parent / roc_csv_filename
        roc_df = pd.DataFrame({
            'FPR': mean_fpr,
            'TPR': mean_tpr,
            'Macro_AUC': macro_auc
        })
        roc_df.to_csv(roc_csv_path, index=False)
        
        print(f"    ✓ Macro ROC: {output_path.name}")
        print(f"    ✓ ROC data: {roc_csv_path.name}")
        
        return mean_fpr, mean_tpr, macro_auc, auc_per_class
    
    def plot_per_class_roc_curves(self, y_true, y_pred_proba, class_names,
                                 title, output_path, dpi=300):
        """Plot per-class (One-vs-Rest) ROC curves."""
        
        n_classes = len(class_names)
        y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))
        
        fig, axes = plt.subplots(1, n_classes, figsize=(6*n_classes, 6), dpi=dpi)
        if n_classes == 1:
            axes = [axes]
        
        colors_palette = sns.color_palette("husl", n_classes)
        
        for i, class_name in enumerate(class_names):
            ax = axes[i]
            
            if y_true_bin[:, i].sum() == 0:
                ax.text(0.5, 0.5, f'No positive samples\nfor {class_name}',
                       ha='center', va='center', fontsize=12, transform=ax.transAxes)
                continue
            
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
            roc_auc = auc(fpr, tpr)
            
            ax.plot(fpr, tpr, color=colors_palette[i], linewidth=3,
                   label=f'{class_name}\n(AUC = {roc_auc:.3f})')
            ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Chance')
            
            ax.set_xlim([0.0, 1.0])
            ax.set_ylim([0.0, 1.05])
            ax.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
            ax.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
            ax.set_title(f'ROC: {class_name} (One-vs-Rest)', fontsize=12, fontweight='bold')
            ax.legend(loc='lower right', fontsize=11)
            ax.grid(True, alpha=0.3)
        
        plt.suptitle(title, fontsize=14, fontweight='bold', y=1.00)
        plt.tight_layout()
        plt.savefig(output_path, dpi=dpi, bbox_inches='tight')
        plt.close()
        
        print(f"    ✓ Per-class ROC: {output_path.name}")

# ============================================================================
# ======================== CONFIGURATION SECTION ============================
# ============================================================================

# --- INPUT/OUTPUT ---
# Replace path with data file location

INPUT_FILE = "data/input.xlsx"
EXTERNAL_VALIDATION_FILE = "data/38-experimental-validation.xlsx"
PROTVEC_FILE = "data/protVec_100d_3grams.csv"
OUTPUT_DIR = Path("data/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- DATA COLUMNS ---
SEQUENCE_COL = "sequence"
TARGET_COL = "category"
LIPIDATION_COL = "lipidation"
METADATA_COLS = ["source_file", "lipidation"]

# --- DATA FILTERING ---
EXCLUDE_UNKNOWN = True
UNKNOWN_KEYWORDS = ["unknown", "nan", "none", ""]

# --- LIPIDATION FILTER ---
LIPIDATION_FILTER = "all"

# --- MODEL PARAMETERS ---
TEST_SIZE = 0.2
RANDOM_STATE = 42
CV_SPLITS = 5
CLASS_WEIGHT = "balanced"

# --- PLOTTING ---
FIGURE_SIZE = (8, 6)
DPI = 300
FONT_SIZE = 12
COLORS_CLASSES = sns.color_palette("husl", 10)

date_str = datetime.now().strftime('%Y-%m-%d_%H%M%S')
print(f"Script started: {date_str}\n")

# ============================================================================
# ======================== 1. LOAD DATA =====================================
# ============================================================================

print("[1] Loading data...")
df = pd.read_excel(INPUT_FILE)
print(f"Loaded {len(df)} sequences from training file (raw)")

df_external = pd.read_excel(EXTERNAL_VALIDATION_FILE)
print(f"Loaded {len(df_external)} sequences from external validation file (raw)")
print(f"Columns: {df.columns.tolist()}\n")

# ============================================================================
# ======================== 2. FILTER DATA ===================================
# ============================================================================

print("[2] Applying filters to training data...")

if EXCLUDE_UNKNOWN:
    print(f"   • Excluding unknown groups...")
    initial_count = len(df)
    mask = ~df[TARGET_COL].astype(str).str.lower().isin(UNKNOWN_KEYWORDS)
    df = df[mask].reset_index(drop=True)
    excluded_count = initial_count - len(df)
    print(f"     Excluded {excluded_count} unknown entries")

print(f"   • Applying lipidation filter: {LIPIDATION_FILTER}")

if LIPIDATION_FILTER == "lipidated":
    df = df[df[LIPIDATION_COL].notna() & (df[LIPIDATION_COL].str.strip() != "")].reset_index(drop=True)
    print(f"     Selected only lipidated peptides")
elif LIPIDATION_FILTER == "non_lipidated":
    df = df[(df[LIPIDATION_COL].isna()) | (df[LIPIDATION_COL].str.strip() == "")].reset_index(drop=True)
    print(f"     Selected only non-lipidated peptides")
else:
    print(f"     Using all data (no lipidation filtering)")

print(f"\nTraining dataset after filtering: {len(df)} sequences")
print(f"Target distribution:\n{df[TARGET_COL].value_counts()}\n")

# ---- Filter external validation data similarly ----
print("[2b] Applying same filters to external validation data...")

if EXCLUDE_UNKNOWN:
    initial_count = len(df_external)
    mask = ~df_external[TARGET_COL].astype(str).str.lower().isin(UNKNOWN_KEYWORDS)
    df_external = df_external[mask].reset_index(drop=True)
    excluded_count = initial_count - len(df_external)
    print(f"   Excluded {excluded_count} unknown entries")

if LIPIDATION_FILTER == "lipidated":
    df_external = df_external[df_external[LIPIDATION_COL].notna() & 
                              (df_external[LIPIDATION_COL].str.strip() != "")].reset_index(drop=True)
elif LIPIDATION_FILTER == "non_lipidated":
    df_external = df_external[(df_external[LIPIDATION_COL].isna()) | 
                              (df_external[LIPIDATION_COL].str.strip() == "")].reset_index(drop=True)

print(f"\nExternal validation dataset after filtering: {len(df_external)} sequences")
print(f"Target distribution:\n{df_external[TARGET_COL].value_counts()}\n")

# ============================================================================
# ======================== 3. LOAD PROTVEC ===================================
# ============================================================================

print("[3] Loading ProtVec 3-gram embeddings...")
protvec_df = pd.read_csv(PROTVEC_FILE, sep='\t')
print(f"Loaded {len(protvec_df)} 3-grams with 100 dimensions\n")

# ============================================================================
# ======================== 4. GENERATE PROTVEC FEATURES =====================
# ============================================================================

print("[4] Converting sequences to ProtVec features...")

def seq_to_protvec(seq, protvec_df, dim=100):
    """Convert amino acid sequence to 100-d ProtVec vector (sum of 3-grams)."""
    vec = np.zeros(dim)
    if len(seq) < 3:
        return vec
    for i in range(len(seq) - 2):
        trigram = seq[i:i+3]
        row = protvec_df[protvec_df['words'] == trigram]
        if len(row) > 0:
            vec += row.iloc[0, 1:].values.astype(float)
    return vec

# Generate vectors for training data
protvec_features = np.array([seq_to_protvec(s, protvec_df) for s in df[SEQUENCE_COL]])
print(f"Generated ProtVec shape (training): {protvec_features.shape}")

# Generate vectors for external validation data
protvec_features_ext = np.array([seq_to_protvec(s, protvec_df) for s in df_external[SEQUENCE_COL]])
print(f"Generated ProtVec shape (external): {protvec_features_ext.shape}\n")

feature_cols = [f"vec{i+1}" for i in range(100)]
X = pd.DataFrame(protvec_features, columns=feature_cols, index=df.index)
X_external = pd.DataFrame(protvec_features_ext, columns=feature_cols, index=df_external.index)

# Target labels
y = pd.factorize(df[TARGET_COL])[0]
class_labels = pd.factorize(df[TARGET_COL])[1]
class_mapping = dict(zip(range(len(class_labels)), class_labels))

# External validation targets
y_external = pd.factorize(df_external[TARGET_COL])[0]

print(f"Class mapping: {class_mapping}")
print(f"Number of classes: {len(class_mapping)}\n")

# ============================================================================
# ======================== 5. TRAIN/TEST SPLIT ==============================
# ============================================================================

print("[5] Train/test split (80/20)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train)}, Internal Test: {len(X_test)}")
print(f"External Validation: {len(X_external)}\n")

# ============================================================================
# ======================== 6. SCALE FEATURES ================================
# ============================================================================

print("[6] Standardizing features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_external_scaled = scaler.transform(X_external)
X_scaled = scaler.transform(X)
print("Done.\n")

# ============================================================================
# ======================== 7. TRAIN MULTIPLE MODELS ==========================
# ============================================================================

print("[7] Training multiple models...\n")

models = {}
predictions = {}
probabilities = {}
metrics_data = []

# ---- 7A: LOGISTIC REGRESSION ----
print("  • Logistic Regression...")
try:
    param_grid_lr = {'C': np.logspace(-3, 3, 20)}
    clf_lr = LogisticRegression(penalty='l1', solver='saga', class_weight=CLASS_WEIGHT, 
                                 max_iter=5000, random_state=RANDOM_STATE, n_jobs=-1)
    grid_lr = GridSearchCV(clf_lr, param_grid_lr, cv=5, scoring='f1_macro', n_jobs=-1)
    grid_lr.fit(X_train_scaled, y_train)
    models['LogisticRegression'] = grid_lr.best_estimator_
    print(f"    Best C: {grid_lr.best_params_['C']:.4f}, CV F1: {grid_lr.best_score_:.4f}")
except Exception as e:
    print(f"    Failed: {e}")

# ---- 7B: RANDOM FOREST ----
print("  • Random Forest...")
try:
    clf_rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, 
                                     class_weight=CLASS_WEIGHT, n_jobs=-1, max_depth=15)
    clf_rf.fit(X_train_scaled, y_train)
    models['RandomForest'] = clf_rf
    print(f"    Fitted.")
except Exception as e:
    print(f"    Failed: {e}")

# ---- 7C: GRADIENT BOOSTING ----
print("  • Gradient Boosting...")
try:
    clf_gb = GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                       max_depth=5, learning_rate=0.1)
    clf_gb.fit(X_train_scaled, y_train)
    models['GradientBoosting'] = clf_gb
    print(f"    Fitted.")
except Exception as e:
    print(f"    Failed: {e}")

# ---- 7D: SVM ----
print("  • Support Vector Machine...")
try:
    clf_svm = SVC(kernel='rbf', C=1.0, probability=True, class_weight=CLASS_WEIGHT,
                  random_state=RANDOM_STATE)
    clf_svm.fit(X_train_scaled, y_train)
    models['SVM'] = clf_svm
    print(f"    ✓ Fitted successfully.")
except Exception as e:
    print(f"    ❌ Failed: {e}")
    
except ValueError as ve:
    print(f"    ❌ ValueError: {ve}")
    print(f"    Trying alternative SVM configuration...")
    try:
        clf_svm = SVC(kernel='linear', C=1.0, probability=True, class_weight=CLASS_WEIGHT,
                      random_state=RANDOM_STATE, n_jobs=1)
        clf_svm.fit(X_train_scaled, y_train)
        models['SVM'] = clf_svm
        print(f"    ✓ Fitted with linear kernel instead.")
    except Exception as e2:
        print(f"    ❌ Also failed: {e2}")
        
except MemoryError as me:
    print(f"    ❌ MemoryError: Dataset too large for SVM with RBF kernel")
    print(f"    Trying linear kernel with smaller C...")
    try:
        clf_svm = SVC(kernel='linear', C=0.1, probability=False, class_weight=CLASS_WEIGHT,
                      random_state=RANDOM_STATE, n_jobs=1)
        clf_svm.fit(X_train_scaled, y_train)
        models['SVM'] = clf_svm
        print(f"    ✓ Fitted with linear kernel and reduced C.")
    except Exception as e2:
        print(f"    ❌ Also failed: {e2}")
        
except Exception as e:
    print(f"    ❌ Failed: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

# ---- 7E: LIGHTGBM ----
if HAS_LGBM:
    print("  • LightGBM...")
    try:
        clf_lgbm = lgb.LGBMClassifier(n_estimators=200, num_leaves=31, learning_rate=0.05,
                                      class_weight='balanced', random_state=RANDOM_STATE,
                                      n_jobs=-1, verbose=-1, is_unbalance=True)
        clf_lgbm.fit(X_train_scaled, y_train)
        models['LightGBM'] = clf_lgbm
        print(f"    Fitted.")
    except Exception as e:
        print(f"    Failed: {e}")

# ---- 7F: XGBOOST ----
if HAS_XGB:
    print("  • XGBoost...")
    try:
        clf_xgb = xgb.XGBClassifier(n_estimators=100, random_state=RANDOM_STATE, 
                                    scale_pos_weight=1, verbose=0, n_jobs=-1)
        clf_xgb.fit(X_train_scaled, y_train)
        models['XGBoost'] = clf_xgb
        print(f"    Fitted.")
    except Exception as e:
        print(f"    Failed: {e}")

print()

# ============================================================================
# ======================== 8. EVALUATE ALL MODELS (INTERNAL TEST) ===========
# ============================================================================

print("[8] Evaluating models on INTERNAL TEST set...\n")
print(f"Number of models trained: {len(models)}")
print(f"Models: {list(models.keys())}\n")

best_model = None
best_accuracy = 0

for model_name, model in models.items():
    print(f"\n{'='*70}")
    print(f"Model: {model_name}")
    print('='*70)
    
    # Internal test predictions
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)
    
    predictions[model_name] = y_pred
    probabilities[model_name] = y_proba
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    # ROC-AUC
    try:
        if len(np.unique(y)) == 2:
            roc_auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
    except:
        roc_auc = 0.0
    
    print(f"[INTERNAL TEST]")
    print(f"  Accuracy:           {acc:.4f}")
    print(f"  Balanced Accuracy:  {bal_acc:.4f}")
    print(f"  Precision:          {prec:.4f}")
    print(f"  Recall:             {rec:.4f}")
    print(f"  F1-Score:           {f1:.4f}")
    print(f"  MCC:                {mcc:.4f}")
    print(f"  ROC-AUC:            {roc_auc:.4f}")
    
    # External validation predictions
    y_pred_ext = model.predict(X_external_scaled)
    y_proba_ext = model.predict_proba(X_external_scaled)
    
    acc_ext = accuracy_score(y_external, y_pred_ext)
    bal_acc_ext = balanced_accuracy_score(y_external, y_pred_ext)
    prec_ext = precision_score(y_external, y_pred_ext, average='macro', zero_division=0)
    rec_ext = recall_score(y_external, y_pred_ext, average='macro', zero_division=0)
    f1_ext = f1_score(y_external, y_pred_ext, average='macro', zero_division=0)
    mcc_ext = matthews_corrcoef(y_external, y_pred_ext)
    
    try:
        if len(np.unique(y_external)) == 2:
            roc_auc_ext = roc_auc_score(y_external, y_proba_ext[:, 1])
        else:
            roc_auc_ext = roc_auc_score(y_external, y_proba_ext, multi_class='ovr', average='macro')
    except:
        roc_auc_ext = 0.0
    
    print(f"\n[EXTERNAL VALIDATION]")
    print(f"  Accuracy:           {acc_ext:.4f}")
    print(f"  Balanced Accuracy:  {bal_acc_ext:.4f}")
    print(f"  Precision:          {prec_ext:.4f}")
    print(f"  Recall:             {rec_ext:.4f}")
    print(f"  F1-Score:           {f1_ext:.4f}")
    print(f"  MCC:                {mcc_ext:.4f}")
    print(f"  ROC-AUC:            {roc_auc_ext:.4f}")
    
    # Store metrics
    metrics_data.append({
        'Model': model_name,
        'Acc_Internal': acc,
        'BalAcc_Internal': bal_acc,
        'Prec_Internal': prec,
        'Rec_Internal': rec,
        'F1_Internal': f1,
        'MCC_Internal': mcc,
        'AUC_Internal': roc_auc,
        'Acc_External': acc_ext,
        'BalAcc_External': bal_acc_ext,
        'Prec_External': prec_ext,
        'Rec_External': rec_ext,
        'F1_External': f1_ext,
        'MCC_External': mcc_ext,
        'AUC_External': roc_auc_ext
    })
    
    # Track best model (based on internal test accuracy)
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = (model_name, model, y_pred, y_proba, y_pred_ext, y_proba_ext)

print(f"\n{'='*70}\n")

# ---- Create metrics comparison table ----
metrics_df = pd.DataFrame(metrics_data).sort_values('Acc_Internal', ascending=False)

# DEBUG: Check if SVM is in the dataframe
print("\n[DEBUG] Models in metrics_df:")
for idx, row in metrics_df.iterrows():
    print(f"  {row['Model']:20} | Acc: {row['Acc_Internal']:.4f} | AUC: {row['AUC_Internal']:.4f}")
print("\n📊 MODEL PERFORMANCE OVERVIEW (INTERNAL TEST vs EXTERNAL VALIDATION)")
print("="*150)
print(metrics_df.to_string(index=False))
print("="*150)

# Save metrics comparison
metrics_df.to_excel(OUTPUT_DIR / f"model_comparison_internal_vs_external_{date_str}.xlsx", index=False)
metrics_df.to_csv(OUTPUT_DIR / f"model_comparison_internal_vs_external_{date_str}.csv", index=False)
print(f"✓ Model comparison saved (.xlsx and .csv)\n")

# Use best model for detailed analysis
best_model_name, best_clf, y_pred_test, y_pred_proba_test, y_pred_ext, y_pred_proba_ext = best_model
print(f"🏆 Best Model (based on internal test): {best_model_name} (Accuracy: {best_accuracy:.4f})\n")

# ============================================================================
# ======================== 9. DETAILED VISUALIZATIONS =======================
# ============================================================================

print("[9] Creating detailed visualizations...\n")

roc_viz = ROCVisualization()
class_names = [class_labels[i] for i in range(len(class_labels))]

def create_confusion_matrix_plot(y_true, y_pred, class_labels, title, output_path, dpi=300):
    """Helper function to create confusion matrix plots"""
    cm = confusion_matrix(y_true, y_pred)
    
    # Counts
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=dpi)
    
    # Counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
                xticklabels=class_labels, yticklabels=class_labels,
                annot_kws={"size": FONT_SIZE+2}, linewidths=2)
    axes[0].set_xlabel("Predicted", fontsize=FONT_SIZE+2, fontweight='bold')
    axes[0].set_ylabel("True", fontsize=FONT_SIZE+2, fontweight='bold')
    axes[0].set_title(f"{title} (Counts)", fontsize=FONT_SIZE+3, fontweight='bold')
    
    # Percentages
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', cbar=True, ax=axes[1],
                xticklabels=class_labels, yticklabels=class_labels,
                annot_kws={"size": FONT_SIZE+2}, cbar_kws={"label": "Percentage (%)"}, linewidths=2)
    axes[1].set_xlabel("Predicted", fontsize=FONT_SIZE+2, fontweight='bold')
    axes[1].set_ylabel("True", fontsize=FONT_SIZE+2, fontweight='bold')
    axes[1].set_title(f"{title} (%)", fontsize=FONT_SIZE+3, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    
    return cm

# ---- 9A & 9B: INTERNAL TEST Confusion Matrices ----
print("  Creating confusion matrices (internal test)...")
cm_internal = create_confusion_matrix_plot(
    y_test, y_pred_test, class_names,
    f"Confusion Matrix - Internal Test\n{best_model_name}",
    OUTPUT_DIR / f"01_confusion_matrix_internal_test_{date_str}.png",
    dpi=DPI
)
print("✓ Internal test confusion matrices")

# ---- 9C & 9D: EXTERNAL VALIDATION Confusion Matrices ----
print("  Creating confusion matrices (external validation)...")
cm_external = create_confusion_matrix_plot(
    y_external, y_pred_ext, class_names,
    f"Confusion Matrix - External Validation\n{best_model_name}",
    OUTPUT_DIR / f"02_confusion_matrix_external_validation_{date_str}.png",
    dpi=DPI
)
print("✓ External validation confusion matrices")

# ---- 9E: INTERNAL TEST Macro ROC ----
print("  Creating macro ROC curves (internal test)...")
fpr_int, tpr_int, auc_int, auc_per_class_int = roc_viz.plot_macro_roc_curve(y_test, y_pred_proba_test, class_names,
    f'Macro-Averaged ROC Curve - Internal Test\n{best_model_name}',
    OUTPUT_DIR / f"03_roc_macro_internal_test_{date_str}.png",
    dpi=DPI
)
print("✓ Internal test macro ROC curve")

# ---- 9F: EXTERNAL VALIDATION Macro ROC ----
print("  Creating macro ROC curves (external validation)...")
fpr_ext, tpr_ext, auc_ext, auc_per_class_ext = roc_viz.plot_macro_roc_curve(
    y_external, y_pred_proba_ext, class_names,
    f'Macro-Averaged ROC Curve - External Validation\n{best_model_name}',
    OUTPUT_DIR / f"04_roc_macro_external_validation_{date_str}.png",
    dpi=DPI
)
print("✓ External validation macro ROC curve")

# Save ROC data to CSV
roc_data_int = pd.DataFrame({
    'FPR': fpr_int,
    'TPR': tpr_int,
    'Macro_AUC': auc_int
})
roc_data_int.to_csv(OUTPUT_DIR / f"roc_macro_internal_test_{date_str}.csv", index=False)
print(f"✓ ROC curve data saved: roc_macro_internal_test_{date_str}.csv")

roc_data_ext = pd.DataFrame({
    'FPR': fpr_ext,
    'TPR': tpr_ext,
    'Macro_AUC': auc_ext
})
roc_data_ext.to_csv(OUTPUT_DIR / f"roc_macro_external_validation_{date_str}.csv", index=False)
print(f"✓ ROC curve data saved: roc_macro_external_validation_{date_str}.csv")

# ---- 9G: Per-class ROC Curves (Internal) ----
print("  Creating per-class ROC curves (internal test)...")
roc_viz.plot_per_class_roc_curves(
    y_test, y_pred_proba_test, class_names,
    f'Per-Class ROC Curves - Internal Test\n{best_model_name}',
    OUTPUT_DIR / f"05_roc_per_class_internal_test_{date_str}.png",
    dpi=DPI
)
print("✓ Per-class ROC curves (internal test)")

# ---- 9H: Per-class ROC Curves (External) ----
print("  Creating per-class ROC curves (external validation)...")
roc_viz.plot_per_class_roc_curves(
    y_external, y_pred_proba_ext, class_names,
    f'Per-Class ROC Curves - External Validation\n{best_model_name}',
    OUTPUT_DIR / f"06_roc_per_class_external_validation_{date_str}.png",
    dpi=DPI
)
print("✓ Per-class ROC curves (external validation)")

# ---- 9I: Model Comparison - Metrics Bar Plot ----
print("  Creating model comparison plots...")
fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=DPI)

# Plot 1: Accuracy Metrics
ax = axes[0, 0]
x = np.arange(len(metrics_df))
width = 0.19

metrics_int = ['Acc_Internal', 'BalAcc_Internal', 'Prec_Internal', 'Rec_Internal', 'F1_Internal']
labels = ['Accuracy', 'Bal. Acc', 'Precision', 'Recall', 'F1-Score']
colors_bar = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for idx, (metric, label, color) in enumerate(zip(metrics_int, labels, colors_bar)):
    ax.bar(x + idx*width, metrics_df[metric], width, label=label, alpha=0.85, edgecolor='black', color=color)

ax.set_ylabel('Score', fontsize=FONT_SIZE+2, fontweight='bold')
ax.set_title('Internal Test - Accuracy Metrics', fontsize=FONT_SIZE+3, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(metrics_df['Model'], rotation=45, ha='right', fontsize=10)
ax.legend(fontsize=FONT_SIZE-1, loc='lower right', ncol=2)
ax.set_ylim([0, 1.1])
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: AUC and MCC
ax = axes[0, 1]
x = np.arange(len(metrics_df))
width = 0.25

ax.bar(x - width/2, metrics_df['AUC_Internal'], width, label='AUC', alpha=0.85, edgecolor='black', color='#e74c3c')
ax.bar(x + width/2, metrics_df['MCC_Internal'], width, label='MCC', alpha=0.85, edgecolor='black', color='#2ca02c')

ax.set_ylabel('Score', fontsize=FONT_SIZE+2, fontweight='bold')
ax.set_title('Internal Test - AUC & MCC', fontsize=FONT_SIZE+3, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['Model'], rotation=45, ha='right', fontsize=10)
ax.legend(fontsize=FONT_SIZE)
ax.set_ylim([0, 1.1])
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: External Validation Metrics
ax = axes[1, 0]
x = np.arange(len(metrics_df))
width = 0.19

metrics_ext = ['Acc_External', 'BalAcc_External', 'Prec_External', 'Rec_External', 'F1_External']

for idx, (metric, label, color) in enumerate(zip(metrics_ext, labels, colors_bar)):
    ax.bar(x + idx*width, metrics_df[metric], width, label=label, alpha=0.85, edgecolor='black', color=color)

ax.set_ylabel('Score', fontsize=FONT_SIZE+2, fontweight='bold')
ax.set_title('External Validation - Accuracy Metrics', fontsize=FONT_SIZE+3, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(metrics_df['Model'], rotation=45, ha='right', fontsize=10)
ax.legend(fontsize=FONT_SIZE-1, loc='lower right', ncol=2)
ax.set_ylim([0, 1.1])
ax.grid(True, alpha=0.3, axis='y')

# Plot 4: External AUC and MCC
ax = axes[1, 1]
x = np.arange(len(metrics_df))
width = 0.25

ax.bar(x - width/2, metrics_df['AUC_External'], width, label='AUC', alpha=0.85, edgecolor='black', color='#e74c3c')
ax.bar(x + width/2, metrics_df['MCC_External'], width, label='MCC', alpha=0.85, edgecolor='black', color='#2ca02c')

ax.set_ylabel('Score', fontsize=FONT_SIZE+2, fontweight='bold')
ax.set_xlabel('Model', fontsize=FONT_SIZE+2, fontweight='bold')
ax.set_title('External Validation - AUC & MCC', fontsize=FONT_SIZE+3, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['Model'], rotation=45, ha='right', fontsize=10)
ax.legend(fontsize=FONT_SIZE)
ax.set_ylim([0, 1.1])
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Multi-Model Performance Comparison (ProtVec)', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"07_model_comparison_multi_plot_{date_str}.png", dpi=DPI, bbox_inches='tight')
plt.close()
print("✓ Model comparison plots")

# ---- 9J: Per-class Metrics Tables ----
print("  Creating per-class metrics tables...")
per_class_metrics_int = []
for i, class_label in enumerate(class_names):
    y_test_binary = (y_test == i).astype(int)
    y_pred_binary = (y_pred_test == i).astype(int)
    
    acc = accuracy_score(y_test_binary, y_pred_binary)
    prec = precision_score(y_test_binary, y_pred_binary, zero_division=0)
    rec = recall_score(y_test_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_test_binary, y_pred_binary, zero_division=0)
    
    per_class_metrics_int.append({
        'Class': str(class_label),
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })

per_class_df_int = pd.DataFrame(per_class_metrics_int)

# ---- Per-class Metrics (External Validation) ----
per_class_metrics_ext = []
for i, class_label in enumerate(class_names):
    y_ext_binary = (y_external == i).astype(int)
    y_pred_binary = (y_pred_ext == i).astype(int)
    
    acc = accuracy_score(y_ext_binary, y_pred_binary)
    prec = precision_score(y_ext_binary, y_pred_binary, zero_division=0)
    rec = recall_score(y_ext_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_ext_binary, y_pred_binary, zero_division=0)
    
    per_class_metrics_ext.append({
        'Class': str(class_label),
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })

per_class_df_ext = pd.DataFrame(per_class_metrics_ext)

# ---- Plot Per-class Metrics ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=DPI)

for ax_idx, (per_class_df, title_text) in enumerate([(per_class_df_int, "Internal Test"), 
                                                       (per_class_df_ext, "External Validation")]):
    ax = axes[ax_idx]
    x = np.arange(len(per_class_df))
    width = 0.2
    
    ax.bar(x - 1.5*width, per_class_df['Accuracy'], width, label='Accuracy', 
           alpha=0.85, edgecolor='black', color='#1f77b4')
    ax.bar(x - 0.5*width, per_class_df['Precision'], width, label='Precision', 
           alpha=0.85, edgecolor='black', color='#ff7f0e')
    ax.bar(x + 0.5*width, per_class_df['Recall'], width, label='Recall', 
           alpha=0.85, edgecolor='black', color='#2ca02c')
    ax.bar(x + 1.5*width, per_class_df['F1-Score'], width, label='F1-Score', 
           alpha=0.85, edgecolor='black', color='#d62728')
    
    ax.set_ylabel('Score', fontsize=FONT_SIZE+2, fontweight='bold')
    ax.set_xlabel('Class', fontsize=FONT_SIZE+2, fontweight='bold')
    ax.set_title(f'Per-Class Metrics ({title_text})\n{best_model_name}', 
                 fontsize=FONT_SIZE+3, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(per_class_df['Class'], fontsize=11)
    ax.legend(fontsize=FONT_SIZE)
    ax.set_ylim([0, 1.1])
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"08_per_class_metrics_comparison_{date_str}.png", dpi=DPI)
plt.close()
print("✓ Per-class metrics plots")

print("\nVisualizations completed.\n")

# ---- 9K: AUROC COMPARISON PLOT WITH BEST MODEL SUMMARY ----
print("  Creating comprehensive model comparison plot...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=DPI)

# ---- Subplot 1: Accuracy Metrics ----
ax = axes[0, 0]
x = np.arange(len(metrics_df))
width = 0.19

metrics_int = ['Acc_Internal', 'BalAcc_Internal', 'Prec_Internal', 'Rec_Internal']
labels = ['Accuracy', 'Bal. Acc', 'Precision', 'Recall']
colors_bar = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for idx, (metric, label, color) in enumerate(zip(metrics_int, labels, colors_bar)):
    ax.bar(x + idx*width - 1.5*width, metrics_df[metric], width, label=label, 
           alpha=0.8, edgecolor='black', color=color)

ax.set_ylabel('Score', fontsize=FONT_SIZE+2, fontweight='bold')
ax.set_title('Accuracy Metrics (Internal Test)', fontsize=FONT_SIZE+3, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['Model'], rotation=45, ha='right', fontsize=10)
ax.legend(fontsize=FONT_SIZE-1, loc='upper right', ncol=2)
ax.set_ylim([0, 1.1])
ax.grid(True, alpha=0.3, axis='y')

# ---- Subplot 2: F1-Score and MCC ----
ax = axes[0, 1]
x = np.arange(len(metrics_df))
width = 0.25

ax.bar(x - width/2, metrics_df['F1_Internal'], width, label='F1-Score', 
       alpha=0.8, edgecolor='black', color='#2ecc71')
ax.bar(x + width/2, metrics_df['MCC_Internal'], width, label='MCC', 
       alpha=0.8, edgecolor='black', color='#f39c12')

ax.set_ylabel('Score', fontsize=FONT_SIZE+2, fontweight='bold')
ax.set_title('F1-Score and MCC (Internal Test)', fontsize=FONT_SIZE+3, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['Model'], rotation=45, ha='right', fontsize=10)
ax.legend(fontsize=FONT_SIZE)
ax.set_ylim([0, 1.1])
ax.grid(True, alpha=0.3, axis='y')

# ---- Subplot 3: AUROC Comparison ----
ax = axes[1, 0]
x = np.arange(len(metrics_df))

# Color code: blue if > 0.7, red if <= 0.7
colors_auc = ['#3498db' if val > 0.7 else '#e74c3c' for val in metrics_df['AUC_Internal']]
bars = ax.bar(x, metrics_df['AUC_Internal'], alpha=0.8, edgecolor='black', 
              linewidth=2, color=colors_auc)

# Add value labels on bars
for bar, val in zip(bars, metrics_df['AUC_Internal']):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.02, f'{val:.3f}',
            ha='center', va='bottom', fontweight='bold', fontsize=FONT_SIZE)

ax.set_ylabel('AUROC', fontsize=FONT_SIZE+2, fontweight='bold')
ax.set_xlabel('Model', fontsize=FONT_SIZE+2, fontweight='bold')
ax.set_title('AUROC Comparison (Internal Test)', fontsize=FONT_SIZE+3, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['Model'], rotation=45, ha='right', fontsize=10)
ax.set_ylim([0, 1.1])
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)

# ---- Subplot 4: Best Model Summary ----
ax = axes[1, 1]
ax.axis('off')

# Find best model by accuracy
best_idx = metrics_df['Acc_Internal'].idxmax()
best_model_row = metrics_df.iloc[best_idx]

summary_text = (
    f"BEST MODEL SUMMARY\n"
    f"{'='*45}\n\n"
    f"Model: {best_model_row['Model']}\n\n"
    f"INTERNAL TEST SET:\n"
    f"  Accuracy:          {best_model_row['Acc_Internal']:.4f}\n"
    f"  Balanced Accuracy: {best_model_row['BalAcc_Internal']:.4f}\n"
    f"  Precision:         {best_model_row['Prec_Internal']:.4f}\n"
    f"  Recall:            {best_model_row['Rec_Internal']:.4f}\n"
    f"  F1-Score:          {best_model_row['F1_Internal']:.4f}\n"
    f"  MCC:               {best_model_row['MCC_Internal']:.4f}\n"
    f"  AUROC:             {best_model_row['AUC_Internal']:.4f}\n\n"
)

props = dict(boxstyle='round', facecolor='#ffffcc', alpha=0.95, 
             edgecolor='black', linewidth=2.5, pad=1.2)
ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, fontsize=FONT_SIZE-1,
        verticalalignment='top', fontfamily='monospace', bbox=props)

plt.suptitle('Multi-Model Performance Comparison (ProtVec)', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"10_model_comparison_comprehensive_{date_str}.png", 
            dpi=DPI, bbox_inches='tight')
plt.close()
print("✓ Comprehensive model comparison plot (2x2 grid with summary)")

# ============================================================================
# ======================== 10. FEATURE IMPORTANCE ============================
# ============================================================================

print("[10] Feature importance (top ProtVec coefficients)...")

if hasattr(best_clf, 'coef_'):
    if len(np.unique(y)) == 2:
        coef_array = best_clf.coef_[0]
    else:
        coef_array = best_clf.coef_.mean(axis=0)
    
    coef_importance = pd.DataFrame({
        'Feature': feature_cols,
        'Coefficient': coef_array
    })
    
    coef_importance['Abs_Coefficient'] = np.abs(coef_importance['Coefficient'])
    coef_importance = coef_importance.sort_values('Abs_Coefficient', ascending=False)
    
    print("\nTop 15 features:")
    print(coef_importance.head(15))
    
    # Plot top features
    fig, ax = plt.subplots(figsize=(10, 7), dpi=DPI)
    top_features = coef_importance.head(15).sort_values('Coefficient')
    colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in top_features['Coefficient']]
    ax.barh(top_features['Feature'], top_features['Coefficient'], color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax.set_xlabel("Coefficient Value", fontsize=FONT_SIZE+2, fontweight='bold')
    ax.set_ylabel("ProtVec Feature", fontsize=FONT_SIZE+2, fontweight='bold')
    ax.set_title(f"Top 15 ProtVec Features - {best_model_name}", fontsize=FONT_SIZE+3, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.axvline(x=0, color='black', linewidth=0.8)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"09_feature_importance_{date_str}.png", dpi=DPI)
    plt.close()
    
    coef_importance.to_excel(OUTPUT_DIR / f"feature_importance_{date_str}.xlsx", index=False)
    print(f"✓ Feature importance saved.\n")
elif hasattr(best_clf, 'feature_importances_'):
    # For tree-based models
    feature_importance = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': best_clf.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nTop 15 features:")
    print(feature_importance.head(15))
    
    fig, ax = plt.subplots(figsize=(10, 7), dpi=DPI)
    top_features = feature_importance.head(15).sort_values('Importance')
    ax.barh(top_features['Feature'], top_features['Importance'], color='steelblue', alpha=0.7, edgecolor='black', linewidth=1.5)
    ax.set_xlabel("Importance", fontsize=FONT_SIZE+2, fontweight='bold')
    ax.set_ylabel("ProtVec Feature", fontsize=FONT_SIZE+2, fontweight='bold')
    ax.set_title(f"Top 15 ProtVec Features - {best_model_name}", fontsize=FONT_SIZE+3, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"09_feature_importance_{date_str}.png", dpi=DPI)
    plt.close()
    
    feature_importance.to_excel(OUTPUT_DIR / f"feature_importance_{date_str}.xlsx", index=False)
    print(f"✓ Feature importance saved.\n")

# ============================================================================
# ======================== 11. SAVE RESULTS ==================================
# ============================================================================

print("[11] Saving results...\n")

# --- Save predictions to Excel (Full dataset) ---
y_pred_all = best_clf.predict(X_scaled)
y_proba_all = best_clf.predict_proba(X_scaled)

results_df = df.copy()
results_df['predicted_class'] = [class_labels[i] for i in y_pred_all]
results_df['true_class'] = [class_labels[i] for i in y]
results_df['predicted_prob_max'] = y_proba_all.max(axis=1)
results_df['correct_prediction'] = y_pred_all == y

for i, class_label in enumerate(class_labels):
    results_df[f'prob_{class_label}'] = y_proba_all[:, i]

results_df.to_excel(OUTPUT_DIR / f"predictions_training_data_{date_str}.xlsx", index=False)
print(f"✓ Predictions (training data): predictions_training_data_{date_str}.xlsx")

# --- Save predictions (External validation) ---
results_external = df_external.copy()
results_external['predicted_class'] = [class_labels[i] for i in y_pred_ext]
results_external['true_class'] = [class_labels[i] for i in y_external]
results_external['predicted_prob_max'] = y_pred_proba_ext.max(axis=1)
results_external['correct_prediction'] = y_pred_ext == y_external

for i, class_label in enumerate(class_labels):
    results_external[f'prob_{class_label}'] = y_pred_proba_ext[:, i]

results_external.to_excel(OUTPUT_DIR / f"predictions_external_validation_{date_str}.xlsx", index=False)
print(f"✓ Predictions (external validation): predictions_external_validation_{date_str}.xlsx")

# --- Save metrics summary ---
accuracy_test = accuracy_score(y_test, y_pred_test)
balanced_accuracy_test = balanced_accuracy_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test, average='macro', zero_division=0)
recall_test = recall_score(y_test, y_pred_test, average='macro', zero_division=0)
f1_test = f1_score(y_test, y_pred_test, average='macro', zero_division=0)
mcc_test = matthews_corrcoef(y_test, y_pred_test)

accuracy_ext = accuracy_score(y_external, y_pred_ext)
balanced_accuracy_ext = balanced_accuracy_score(y_external, y_pred_ext)
precision_ext = precision_score(y_external, y_pred_ext, average='macro', zero_division=0)
recall_ext = recall_score(y_external, y_pred_ext, average='macro', zero_division=0)
f1_ext = f1_score(y_external, y_pred_ext, average='macro', zero_division=0)
mcc_ext = matthews_corrcoef(y_external, y_pred_ext)

metrics_summary = {
    'Metric': [
        'Best Model',
        '--- INTERNAL TEST SET ---',
        'Accuracy', 'Balanced Accuracy', 'Precision', 'Recall', 'F1-Score', 'MCC', 'ROC-AUC',
        '--- EXTERNAL VALIDATION SET ---',
        'Accuracy', 'Balanced Accuracy', 'Precision', 'Recall', 'F1-Score', 'MCC', 'ROC-AUC',
        '--- DATA INFO ---',
        'N Test Samples', 'N External Samples', 'N Train Samples',
        'N Total Samples', 'N Classes',
        '--- CONFIG ---',
        'Lipidation Filter', 'Exclude Unknown'
    ],
    'Value': [
        best_model_name,
        '',
        f"{accuracy_test:.4f}", f"{balanced_accuracy_test:.4f}", f"{precision_test:.4f}", 
        f"{recall_test:.4f}", f"{f1_test:.4f}", f"{mcc_test:.4f}", f"{auc_int:.4f}",
        '',
        f"{accuracy_ext:.4f}", f"{balanced_accuracy_ext:.4f}", f"{precision_ext:.4f}", 
        f"{recall_ext:.4f}", f"{f1_ext:.4f}", f"{mcc_ext:.4f}", f"{auc_ext:.4f}",
        '',
        len(y_test), len(y_external), len(y_train),
        len(y), len(class_labels),
        '',
        LIPIDATION_FILTER, str(EXCLUDE_UNKNOWN)
    ]
}

metrics_summary_df = pd.DataFrame(metrics_summary)
metrics_summary_df.to_excel(OUTPUT_DIR / f"metrics_summary_{date_str}.xlsx", index=False)
print(f"✓ Metrics summary: metrics_summary_{date_str}.xlsx")

# --- Save per-class metrics ---
per_class_df_int['Dataset'] = 'Internal Test'
per_class_df_int['Model'] = best_model_name
per_class_df_ext['Dataset'] = 'External Validation'
per_class_df_ext['Model'] = best_model_name
per_class_combined = pd.concat([per_class_df_int, per_class_df_ext], ignore_index=True)
per_class_combined.to_excel(OUTPUT_DIR / f"per_class_metrics_{date_str}.xlsx", index=False)
print(f"✓ Per-class metrics: per_class_metrics_{date_str}.xlsx")

# --- Save best model ---
model_file = OUTPUT_DIR / f"model_protvec_{best_model_name}_{date_str}.pkl"
with open(model_file, 'wb') as f:
    pickle.dump({
        'model': best_clf,
        'scaler': scaler,
        'feature_names': feature_cols,
        'class_mapping': class_mapping,
        'class_labels': class_labels.tolist(),
        'model_name': best_model_name,
        'protvec_file': PROTVEC_FILE
    }, f)
print(f"✓ Best model: model_protvec_{best_model_name}_{date_str}.pkl")

# --- Save all model comparison data ---
metrics_df.to_excel(OUTPUT_DIR / f"all_models_comparison_{date_str}.xlsx", index=False)
print(f"✓ All models comparison: all_models_comparison_{date_str}.xlsx")

print()

# ============================================================================
# ======================== 12. FINAL SUMMARY ==================================
# ============================================================================

print("\n" + "="*100)
print("PIPELINE COMPLETED SUCCESSFULLY")
print("="*100)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\n📊 CONFIGURATION:")
print(f"  • Training file: {INPUT_FILE}")
print(f"  • External validation file: {EXTERNAL_VALIDATION_FILE}")
print(f"  • Lipidation filter: {LIPIDATION_FILTER}")
print(f"  • Training dataset size: {len(df)} sequences")
print(f"  • External validation size: {len(df_external)} sequences")
print(f"  • Classes: {', '.join([str(c) for c in class_labels])}")
print(f"  • Models trained: {len(metrics_df)}")

print(f"\n🏆 BEST MODEL: {best_model_name}")

print(f"\n📊 INTERNAL TEST SET RESULTS:")
print(f"  • Accuracy:           {accuracy_test:.4f}")
print(f"  • Balanced Accuracy:  {balanced_accuracy_test:.4f}")
print(f"  • Precision:          {precision_test:.4f}")
print(f"  • Recall:             {recall_test:.4f}")
print(f"  • F1-Score:           {f1_test:.4f}")
print(f"  • MCC:                {mcc_test:.4f}")
print(f"  • ROC-AUC (macro):    {auc_int:.4f}")

print(f"\n📊 EXTERNAL VALIDATION SET RESULTS:")
print(f"  • Accuracy:           {accuracy_ext:.4f}")
print(f"  • Balanced Accuracy:  {balanced_accuracy_ext:.4f}")
print(f"  • Precision:          {precision_ext:.4f}")
print(f"  • Recall:             {recall_ext:.4f}")
print(f"  • F1-Score:           {f1_ext:.4f}")
print(f"  • MCC:                {mcc_ext:.4f}")
print(f"  • ROC-AUC (macro):    {auc_ext:.4f}")

print(f"\n📋 MODEL PERFORMANCE COMPARISON TABLE:")
print(metrics_df.to_string(index=False))

print(f"\n📁 KEY OUTPUT FILES:")
print(f"  ✓ 01_confusion_matrix_internal_test_{date_str}.png")
print(f"  ✓ 02_confusion_matrix_external_validation_{date_str}.png")
print(f"  ✓ 03_roc_macro_internal_test_{date_str}.png (+ CSV)")
print(f"  ✓ 04_roc_macro_external_validation_{date_str}.png (+ CSV)")
print(f"  ✓ 05_roc_per_class_internal_test_{date_str}.png")
print(f"  ✓ 06_roc_per_class_external_validation_{date_str}.png")
print(f"  ✓ 07_model_comparison_multi_plot_{date_str}.png")
print(f"  ✓ 08_per_class_metrics_comparison_{date_str}.png")
print(f"  ✓ 09_feature_importance_{date_str}.png")
print(f"  ✓ 10_model_comparison_comprehensive_{date_str}.png")  # NEW LINE
print(f"  ✓ model_comparison_internal_vs_external_{date_str}.xlsx")
print(f"  ✓ model_comparison_internal_vs_external_{date_str}.csv")
print(f"  ✓ all_models_comparison_{date_str}.xlsx")
print(f"  ✓ predictions_training_data_{date_str}.xlsx")
print(f"  ✓ predictions_external_validation_{date_str}.xlsx")
print(f"  ✓ metrics_summary_{date_str}.xlsx")
print(f"  ✓ per_class_metrics_{date_str}.xlsx")
print(f"  ✓ feature_importance_{date_str}.xlsx")
print(f"  ✓ model_protvec_{best_model_name}_{date_str}.pkl")

print("\n" + "="*100)
print("✅ ANALYSIS COMPLETE")
print("="*100 + "\n")